# Dynamic & Hybrid Conditioning for Compositional Image Retrieval

**Deep Learning Assignment 2026 — CelebA · CLIP ViT-B/32**

> Single self-contained notebook. Mount your Drive, point the paths in the **Config**
> cell at your artifacts, then *Runtime → Run all*. No external project files are
> imported — every method is defined inline.

## Abstract

Given a reference face `v_ref`, a set of positive textual constraints `T⁺` and negative
constraints `T⁻`, we retrieve CelebA test images that **preserve the identity** of `v_ref`
while **satisfying every +/− modification**. We build an escalating ladder of methods on a
**frozen** CLIP ViT-B/32 embedding space and a precomputed image-feature database:

| Tier | Method | One-line idea |
|------|--------|---------------|
| **0** | Vanilla latent arithmetic | `q = v_ref + α(Σt⁺ − Σt⁻)`, cosine rank — the floor. |
| **0+** | Enhanced (3 geometry fixes) | modality-gap centering + delta-norm + prompt ensembling. |
| **1** | CLAY reproduction | manifold-aware SVD subspace from *stacked* prompts — the method-to-beat. |
| **2a-S** | Asymmetric conditional subspaces | separate +/− subspaces; negation as a **subspace penalty**. |
| **2a-V** | Visual-prototype (GDE) | tangent-mean attribute directions; negation as **orthogonal rejection**. |
| **2a-SV** | **Tier-2c** (the synthesis) | V's image-space query rejection + S's `k`-dim subspace, mined from **images**. |

Our contribution (Tier-2a, **training-free**) is the asymmetric treatment of negation:
"−X" is *"any value but X"* (a region of the sphere), **not** vector subtraction toward
"anti-X". This is the gradeable insight the assignment asks for, demonstrated without any
learned parameters — CLIP stays frozen throughout. We push it to its training-free limit with **Tier-2c**, which fuses Tier-2b's image-space query rejection with Tier-2a's multi-dimensional subspace; its convergence with Tier-2b marks the **ceiling of training-free negation** and is what motivates the trained fusion module Φ that follows (§16).


## How this notebook is organized

It reads as a report interleaved with runnable code, in the order the assignment's
deliverable section asks for:

1. **Setup & Config** — Colab mount, dependencies, all paths/knobs in one place.
2. **Background** — CLIP's shared space, the *modality gap*, CLAY's pre-SVD bottleneck, the GDE manifold view.
3. **Data & contract** — the index-not-filename golden rule, the `[N,40]` attribute tensor, the ground-truth JSON.
4. **Sanity checks** — hard assertions that *prove* the indexing/alignment before any score is computed.
5. **Evaluation protocol** — Recall@K (hit-rate) + Precision@K, relaxed-Hamming ground truth.
6. **Frozen feature database** — offline CLIP extraction (image table, attribute text, prompt bank).
7. **Modality-gap analysis** — the geometric fact that motivates every later design choice.
8. **Methods ladder** — Tier 0 → 0+ → 1 → 2a-S → 2a-V → 2a-SV (the synthesis), each with math, code, and a results table.
9. **Results & discussion** — master comparison table, charts, qualitative success/failure grids.
10. **Conclusion** — the training-free ceiling and the pivot to a trained Φ; limitations; references.


---
## 1. Setup & Configuration

### 1.1 Colab session bootstrap (run once)

Mounts Drive, installs the CLIP backbone (`transformers`), and unzips CelebA onto the
runtime's local SSD (fast disk; wiped on disconnect — that's fine, we re-unzip next time).
If you are **not** on Colab, skip these two cells and just make the **Config** paths valid.


In [ ]:
# Colab only — mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Colab only — install the CLIP backbone and stage CelebA on the local SSD.
!pip -q install "transformers>=4.40"
!mkdir -p /content/datasets
# -n = never overwrite (no-op if already unzipped this session).
!unzip -q -n /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/

### 1.2 Config — the single source of truth for paths and knobs

Every path and hyperparameter lives here so nothing is hard-coded deeper in the notebook.
**Edit `DRIVE_ROOT` / `DATASET_ROOT` to match your Drive layout.** Cached `.pt` artifacts
(image features, prompt bank, mined directions) are read from — and written back to —
`ARTIFACTS_DIR`, so a second run is instant.


In [ ]:
from pathlib import Path
import torch, numpy as np, random

# --- paths (edit to match your Drive) ---------------------------------------
DRIVE_ROOT     = Path("/content/drive/MyDrive/datasets")   # your Drive folder
DATASET_ROOT   = Path("/content/datasets")                 # local SSD with the unzipped celeba/
ARTIFACTS_DIR  = DRIVE_ROOT / "artifacts"                  # cached tensors live here (persisted)
EVAL_JSON_PATH = DRIVE_ROOT / "celeba_evaluation.json"     # authoritative ground truth
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# --- fixed by the assignment spec -------------------------------------------
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"   # spec §3.3 — everyone uses this
FEATURE_DIM     = 512                              # CLIP ViT-B/32 projection dim
KS              = (1, 5, 10)                        # report metrics at these cutoffs (spec §3.1.3)

# --- reproducibility ---------------------------------------------------------
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| artifacts:", ARTIFACTS_DIR)

---
## 2. Background & problem framing

**CLIP (Radford et al., 2021)** maps images and text into a shared, L2-normalized
embedding space — a unit hypersphere `S^{511}` — trained so that matching image/text pairs
have high cosine similarity. Because everything lives on the unit sphere, *cosine similarity
= dot product*, and retrieval is a single matrix–vector product against a frozen database.

**The modality gap (Liang et al., 2022).** CLIP does **not** place images and text in one
blob. Empirically, image embeddings occupy a narrow cone and text embeddings a *different*
cone, with a wide empty band between them. A direct consequence: every attribute text vector
`t(a)` is dominated by the same shared "text-ness" component (the text-cone centroid `μ_txt`),
which carries **no** attribute signal. Naïvely adding `t(a)` to an image vector mostly injects
this useless common component — §7 visualizes this, and it is exactly what Tier-0+ corrects.

**CLAY (Lim et al., 2026)** reframes CLIP's space as a *text-conditional similarity space*:
from a set of condition prompts it builds a manifold-aware **subspace** (log-map to the
tangent space at the prompt mean → SVD → top-k right singular vectors) and measures
`reference ↔ database` similarity *inside* that subspace, leaving the visual DB frozen.
Its weakness — the entire point of this assignment — is that with **multiple** conditions it
**naïvely stacks all prompts into one SVD** (the "pre-SVD bottleneck"): no dynamic weighting,
and **no native notion of negation** ("must NOT have X").

**GDE (Berasi et al., 2025)** shows CLIP representations are *geodesically decomposable*:
a concept's primitive direction is the **tangent mean** of the embeddings exhibiting it, and
concepts compose by addition in the tangent space at the corpus mean (then `Exp` back to the
sphere). Tier-2a-V builds directly on this.

**Our task.** Replace the rigid pre-SVD stack with a fusion that treats positives and
negatives *asymmetrically*. The key modeling insight: **negation is not subtraction.**
Subtracting `t_X` overshoots into "anti-X"; the correct reading of "−X" is "*remove the X
axis / penalize energy along it*", leaving any other value of that attribute acceptable. We
realize this two ways, both training-free: as a **subspace penalty** (Tier-2a) and as
**orthogonal rejection** in the tangent space (Tier-2b).


---
## 3. Data & the data contract

### 3.1 The golden rule: images are referenced by dataset **index**, never filename

The ground-truth JSON refers to every image by an integer that means *"the image at that
position in the PyTorch `CelebA` dataset"* — i.e. `celeba[13]`, **not** the file `000013.jpg`.
The torchvision test split is a subsequence of the full dataset, so `celeba.filename[13]`
is actually `"182651.jpg"`. Every index everywhere (rankings, ground truth, the feature
table) is this dataset index. Mixing the two up is the single most common silent failure in
this task, so §4 asserts it before anything else runs.


In [ ]:
# Master list of the 40 CelebA attributes, in the fixed order of list_attr_celeba.txt.
# Column j of every attribute/feature tensor means ATTRIBUTE_NAMES[j].
ATTRIBUTE_NAMES = [
    "5_o_Clock_Shadow", "Arched_Eyebrows", "Attractive", "Bags_Under_Eyes", "Bald",
    "Bangs", "Big_Lips", "Big_Nose", "Black_Hair", "Blond_Hair",
    "Blurry", "Brown_Hair", "Bushy_Eyebrows", "Chubby", "Double_Chin",
    "Eyeglasses", "Goatee", "Gray_Hair", "Heavy_Makeup", "High_Cheekbones",
    "Male", "Mouth_Slightly_Open", "Mustache", "Narrow_Eyes", "No_Beard",
    "Oval_Face", "Pale_Skin", "Pointy_Nose", "Receding_Hairline", "Rosy_Cheeks",
    "Sideburns", "Smiling", "Straight_Hair", "Wavy_Hair", "Wearing_Earrings",
    "Wearing_Hat", "Wearing_Lipstick", "Wearing_Necklace", "Wearing_Necktie", "Young",
]
ATTR_TO_IDX = {name: i for i, name in enumerate(ATTRIBUTE_NAMES)}  # "Smiling" -> 31
assert len(ATTRIBUTE_NAMES) == 40

### 3.2 Load CelebA + build the attribute tensor

CLIP preprocessing is resize-224 + CLIP mean/std normalization (identical to the HF
processor). `shuffle` is never applied anywhere, so `celeba[i]` is stable and row `i` of
every tensor refers to the same image. The attribute tensor is `[N, 40]` `float32` with
values in `{0,1}` (torchvision already maps the raw `-1/+1` labels to `0/1` in `.attr`).


In [ ]:
from torchvision import datasets, transforms

# CLIP ViT-B/32 preprocessing — resize 224 + CLIP mean/std normalization.
_CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
_CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]
CLIP_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=_CLIP_MEAN, std=_CLIP_STD),
])

def load_celeba(split, transform=CLIP_TRANSFORM):
    # CelebA split with CLIP preprocessing; download=False (staged on the SSD already).
    return datasets.CelebA(root=DATASET_ROOT, split=split, download=False, transform=transform)

def build_attributes(celeba):
    # [N, 40] float32 0/1 mask, row i == celeba[i] (CONTRACT §2). torchvision's `.attr`
    # is already 0/1 and column-aligned to celeba.attr_names — we assert that alignment.
    assert list(celeba.attr_names[:40]) == ATTRIBUTE_NAMES, "Attribute column order drift vs master list."
    return celeba.attr.to(torch.float32)

celeba       = load_celeba("test")
attributes   = build_attributes(celeba)
print(f"Test split: {len(celeba)} images | attributes {tuple(attributes.shape)} {attributes.dtype}")

### 3.3 The ground-truth JSON

A list of query objects, each with a `query` string (e.g. `"+Smiling"`) and a `ground_truth`
dict mapping **source index → list of valid target indices**. A target is valid iff it
*strictly* satisfies the +/− constraints **and** sits within Hamming distance ≤ 2 of the
source on the *other* attributes (identity preservation). Only sources with ≥ 5 valid targets
are included. The JSON is **authoritative**: the spec text lists 12 queries but the file has
14 (`-Young` appears twice and two composed queries differ) — we evaluate all 14 as-is.


In [ ]:
import json

def load_eval_json(path):
    # Load the authoritative evaluation JSON (a list of query dicts). Keys are STRING
    # source indices; cast with int(...) at use. We never modify this file.
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

gt_list = load_eval_json(EVAL_JSON_PATH)
print(f"{len(gt_list)} queries:")
for e in gt_list:
    print(f"  {e['query']:<45s}  {len(e['ground_truth'])} sources")

---
## 4. Sanity checks (CRITICAL — run before any scoring)

These are **assertions**, not prints: a broken assumption fails loudly here, before any
method can produce a silently-wrong score. We guard the three things everything downstream
depends on: (1) **indexing** (`celeba.filename[13] == "182651.jpg"`, `N == 19962`),
(2) **attribute tensor** shape/dtype/range/alignment, (3) **ground-truth viability** (every
source has ≥ 5 in-range targets, none equal to its own index).


In [ ]:
EXPECTED_FILENAME_AT_13 = "182651.jpg"   # indexing tripwire (CONTRACT §0)
EXPECTED_N_IMAGES = 19962
MIN_TARGETS_PER_SOURCE = 5

def assert_indexing(celeba):
    # Check 1 — the indexing tripwire. One failure here makes every score silently wrong.
    actual = celeba.filename[13]
    assert actual == EXPECTED_FILENAME_AT_13, (
        f"INDEXING BROKEN: celeba.filename[13] == {actual!r}, expected {EXPECTED_FILENAME_AT_13!r}. "
        "GT indices are PyTorch dataset indices; a reshuffled split or filename lookup crept in.")
    assert len(celeba) == EXPECTED_N_IMAGES, f"Test split has {len(celeba)}, expected {EXPECTED_N_IMAGES}."

def assert_attributes(attributes, celeba):
    # Check 2 — [N,40] float32, values in {0,1}, row-aligned to the dataset.
    assert attributes.dtype == torch.float32 and attributes.shape == (len(celeba), 40)
    assert set(torch.unique(attributes).tolist()) <= {0.0, 1.0}, "attributes must be a 0/1 mask."

def assert_gt_viability(gt_list, n_images=EXPECTED_N_IMAGES):
    # Check 3 — the JSON honours the eval protocol the rest of the code assumes.
    n_sources = 0
    for entry in gt_list:
        assert "query" in entry and "ground_truth" in entry
        for src, targets in entry["ground_truth"].items():
            n_sources += 1
            src_idx = int(src)
            assert 0 <= src_idx < n_images
            assert len(targets) >= MIN_TARGETS_PER_SOURCE, f"{entry['query']} src {src_idx} has < 5 targets."
            for t in targets:
                assert 0 <= int(t) < n_images and int(t) != src_idx
    return n_sources

assert_indexing(celeba)
assert_attributes(attributes, celeba)
n_sources = assert_gt_viability(gt_list)
print(f"[OK] indexing, attributes, and {len(gt_list)} queries / {n_sources} sources all verified — safe to score.")

---
## 5. Evaluation protocol

Metric definitions are taken verbatim from spec §3.1.3 and are the **ruler** every method is
scored through — so all tiers are measured identically.

- **Recall@K (hit-rate, primary):** `1` if the top-K share ≥ 1 image with the ground-truth
  set `G`, else `0`. *(This is a hit-rate, not textbook recall `|hits|/|relevant|`.)*

$$\text{Recall@}K = \mathbb{1}\!\left[\,|R_K \cap G| > 0\,\right]$$

- **Precision@K (secondary):** density of correct matches in the top-K.

$$\text{Precision@}K = \frac{|R_K \cap G|}{K}$$

Both are computed per source, then **averaged over all valid sources** of a query; the headline
**MEAN** row macro-averages over the 14 queries. A *ranking* is a `list[int]` of dataset
indices, best-first, with the source already excluded. Every method plugs in through a
`make_get_ranking(query_str) -> (src_idx -> ranking)` callback.


In [ ]:
import pandas as pd

def recall_at_k(ranking, ground_truth, k):
    # Recall@K (primary) — hit rate: 1.0 if top-K ∩ G ≠ ∅, else 0.0 (spec §3.1.3).
    return 1.0 if (set(ranking[:k]) & ground_truth) else 0.0

def precision_at_k(ranking, ground_truth, k):
    # Precision@K (secondary) — |top-K ∩ G| / K. Denominator is K, not |G|.
    return len(set(ranking[:k]) & ground_truth) / k

def parse_query(query_str):
    # Parse a query into (positive_attrs, negative_attrs). Accepts JSON style
    # ("+Black_Hair, -Wavy_Hair") and spec-table style ("+ Black Hair & - Wavy Hair");
    # names normalize to underscores. Each piece must start with + or -.
    pos, neg = [], []
    for piece in query_str.replace("&", ",").split(","):
        piece = piece.strip()
        if not piece:
            continue
        sign, name = piece[0], piece[1:].strip().replace(" ", "_")
        if sign == "+":   pos.append(name)
        elif sign == "-": neg.append(name)
        else: raise ValueError(f"Query piece must start with + or -: {piece!r}")
    return pos, neg

def evaluate_query(ground_truth, get_ranking, ks=KS):
    # Score one query, averaging each metric over all its valid sources.
    sums = {f"recall@{k}": 0.0 for k in ks}; sums.update({f"precision@{k}": 0.0 for k in ks})
    sources = list(ground_truth.keys())
    for src in sources:
        targets = {int(t) for t in ground_truth[src]}
        ranking = get_ranking(int(src))
        for k in ks:
            sums[f"recall@{k}"]    += recall_at_k(ranking, targets, k)
            sums[f"precision@{k}"] += precision_at_k(ranking, targets, k)
    n = len(sources)
    out = {m: (tot / n if n else 0.0) for m, tot in sums.items()}
    out["num_sources"] = n
    return out

def evaluate_all(gt_list, make_get_ranking, ks=KS):
    # Score every query in the benchmark. make_get_ranking(query_str) yields the per-query
    # callback — the single seam every method plugs into.
    return {e["query"]: evaluate_query(e["ground_truth"], make_get_ranking(e["query"]), ks)
            for e in gt_list}

def mean_metrics(results, ks=KS):
    # Macro-average each metric over the queries (every query weighted equally).
    n = len(results)
    return {**{f"R@{k}": sum(r[f"recall@{k}"] for r in results.values()) / n for k in ks},
            **{f"P@{k}": sum(r[f"precision@{k}"] for r in results.values()) / n for k in ks}}

def results_to_df(results, ks=KS):
    # Per-query table + a macro-MEAN row, as a tidy DataFrame for display.
    rows = [{"query": q,
             **{f"R@{k}": r[f"recall@{k}"] for k in ks},
             **{f"P@{k}": r[f"precision@{k}"] for k in ks},
             "#src": r["num_sources"]} for q, r in results.items()]
    mean = {"query": "MEAN", **mean_metrics(results, ks), "#src": ""}
    return pd.concat([pd.DataFrame(rows), pd.DataFrame([mean])], ignore_index=True)

# Global registry so the §9 master table can compare every tier we run.
ALL_RESULTS = {}
def register(name, results):
    ALL_RESULTS[name] = mean_metrics(results)
    return results

def show(results, title, ks=KS):
    # Print a title, display the per-query table, return results unchanged (for chaining).
    print(title)
    try:
        from IPython.display import display
        display(results_to_df(results, ks).round(4))
    except Exception:
        print(results_to_df(results, ks).round(4).to_string(index=False))
    return results

---
## 6. The frozen feature database (offline, compute-once)

Following CLAY's efficiency backbone, we encode the corpus **once** with frozen CLIP and cache
three tensors in `ARTIFACTS_DIR`; every method then ranks via one matrix–vector product. Each
loader is **load-or-build**: if the `.pt` is already on your Drive it loads instantly,
otherwise it extracts on the GPU and caches. CLIP is never fine-tuned.

- `clip_image_features_test.pt` — `[N, 512]`, L2-normalized, row `i` == `celeba[i]`.
- `clip_attr_text_features.pt` — `[40, 512]`, one prompt per attribute (Tier-0's direction vectors).
- `clip_attr_prompt_bank.pt` — `[40, n, 512]`, a *stack* of ~60 paraphrases per attribute (CLAY's SVD needs spread, not a single vector).


In [ ]:
from torch.utils.data import DataLoader

@torch.no_grad()
def _encode_images(celeba, batch_size=256):
    # Encode an entire split with frozen CLIP → [N, 512], L2-normalized, in strict index
    # order (shuffle=False, so row i == celeba[i]). We call vision_model → visual_projection
    # explicitly rather than get_image_features: the latter's return type varies across
    # transformers versions. Images arrive already CLIP-normalized, so we bypass CLIPProcessor.
    from transformers import CLIPModel
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
    loader = DataLoader(celeba, batch_size=batch_size, shuffle=False, num_workers=2,
                        collate_fn=lambda b: torch.stack([img for img, _ in b]))
    feats, done = [], 0
    for pixel_values in loader:
        out = model.vision_model(pixel_values=pixel_values.to(DEVICE))
        f = model.visual_projection(out.pooler_output)
        feats.append(torch.nn.functional.normalize(f, p=2, dim=1).cpu())
        done += pixel_values.shape[0]; print(f"  encoded {done}/{len(celeba)}", end="\r")
    print()
    return torch.cat(feats).to(torch.float32)

def load_or_extract_image_features(split, n_expected=None, force=False):
    # Load-or-build the [N,512] CLIP image table for a split; verify unit norms + row count.
    path = ARTIFACTS_DIR / f"clip_image_features_{split}.pt"
    if path.exists() and not force:
        feats = torch.load(path, weights_only=True)
        print(f"[OK] loaded {split} image features {tuple(feats.shape)} (cache)")
    else:
        print(f"Extracting {split} image features with frozen CLIP …")
        feats = _encode_images(load_celeba(split))
        torch.save(feats, path); print(f"  saved {path}")
    assert feats.shape[1] == FEATURE_DIM
    assert torch.allclose(feats.norm(dim=1), torch.ones(feats.shape[0]), atol=1e-3), "rows not unit-norm"
    if n_expected is not None:
        assert feats.shape[0] == n_expected, f"{split}: {feats.shape[0]} rows != {n_expected}"
    return feats

image_features = load_or_extract_image_features("test", n_expected=len(celeba))

In [ ]:
@torch.no_grad()
def load_or_extract_attr_text_features(force=False):
    # Load-or-build the [40,512] single-prompt attribute text table (Tier-0 directions).
    # Row j = CLIP("a photo of a person with <attr j>"), L2-normalized. Cheap (40 prompts).
    path = ARTIFACTS_DIR / "clip_attr_text_features.pt"
    if path.exists() and not force:
        t = torch.load(path, weights_only=True); print(f"[OK] loaded attr text {tuple(t.shape)} (cache)")
        return t
    from transformers import CLIPModel, AutoTokenizer
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(CLIP_MODEL_NAME)
    prompts = ["a photo of a person with " + n.replace("_", " ").lower() for n in ATTRIBUTE_NAMES]
    toks = tok(prompts, padding=True, return_tensors="pt").to(DEVICE)
    out = model.text_model(**toks)
    t = torch.nn.functional.normalize(model.text_projection(out.pooler_output), p=2, dim=1).cpu().float()
    torch.save(t, path); print(f"  saved {path} {tuple(t.shape)}")
    return t

attr_text_features = load_or_extract_attr_text_features()

### 6.1 The prompt bank — why CLAY needs more than one sentence per attribute

Tier-0 uses **one** text vector per attribute. CLAY does not: it builds a *subspace* per
condition by SVD on a **stack** of many same-meaning prompts. With `n = 1` the stack is rank-1
and every CLAY lever (subspace dimensionality `k`, spread) is meaningless. CLAY generates these
with an LLM; for reproducibility and course-policy compliance we generate them **deterministically**
from a grid of *sentence frames* × *per-attribute synonyms* (≈ 12 × 5 ⇒ up to 60 prompts/attr).
The downstream geometry is identical — only the *source* of the sentences differs. Phrases are
kept deliberately plain: CLIP ViT-B/32 is a weak language model, so concrete visual words beat
clever paraphrase.


In [ ]:
# Sentence frames — structural spread. Each {phrase} slot is filled with a synonym below.
FRAMES = [
    "a photo of a person with {phrase}", "a photo of a person who has {phrase}",
    "a portrait of someone with {phrase}", "a close-up of a face with {phrase}",
    "an image of a person showing {phrase}", "a headshot of an individual with {phrase}",
    "a picture of a face with {phrase}", "a cropped photo of a person with {phrase}",
    "a selfie of someone with {phrase}", "a candid shot of a person having {phrase}",
    "a frontal portrait of a face with {phrase}", "a clear photograph of a person with {phrase}",
]
# Predicative frames — for adjective-style attributes that read better as "a {phrase} person".
PREDICATIVE_FRAMES = [
    "a photo of a {phrase} person", "a photo of someone who is {phrase}",
    "a portrait of a {phrase} person", "a close-up of a {phrase} face",
    "an image of a {phrase} individual", "a headshot of a {phrase} person",
    "a picture of a {phrase} face", "a cropped photo of a {phrase} person",
    "a selfie of a {phrase} person", "a candid shot of a {phrase} individual",
    "a frontal portrait of a {phrase} person", "a clear photograph of a {phrase} person",
]
# Per-attribute ("noun"|"adj", synonyms) — lexical spread. "noun" -> FRAMES, "adj" -> PREDICATIVE_FRAMES.
ATTR_PHRASES = {
    "5_o_Clock_Shadow": ("noun", ["a five o'clock shadow", "light stubble", "a faint beard shadow", "short stubble on the face", "a day's worth of stubble"]),
    "Arched_Eyebrows": ("noun", ["arched eyebrows", "curved eyebrows", "highly arched brows", "sharply arched eyebrows", "elegantly curved brows"]),
    "Attractive": ("adj", ["attractive", "good looking", "beautiful", "handsome", "striking-looking"]),
    "Bags_Under_Eyes": ("noun", ["bags under the eyes", "puffy under-eyes", "eye bags", "swollen under-eye skin", "dark circles under the eyes"]),
    "Bald": ("adj", ["bald", "hairless on the head", "with no hair", "completely bald", "with a shaved head"]),
    "Bangs": ("noun", ["bangs", "a fringe of hair over the forehead", "hair covering the forehead", "a straight fringe", "front bangs across the brow"]),
    "Big_Lips": ("noun", ["big lips", "full lips", "large lips", "plump lips", "thick full lips"]),
    "Big_Nose": ("noun", ["a big nose", "a large nose", "a prominent nose", "a wide nose", "a broad nose"]),
    "Black_Hair": ("noun", ["black hair", "dark black hair", "jet-black hair", "deep black hair", "raven-black hair"]),
    "Blond_Hair": ("noun", ["blond hair", "blonde hair", "light golden hair", "pale yellow hair", "bright blonde hair"]),
    "Blurry": ("adj", ["blurry", "out of focus", "unsharp", "blurred and hazy", "low in sharpness"]),
    "Brown_Hair": ("noun", ["brown hair", "brunette hair", "dark brown hair", "chestnut-brown hair", "medium brown hair"]),
    "Bushy_Eyebrows": ("noun", ["bushy eyebrows", "thick eyebrows", "dense brows", "heavy bushy eyebrows", "full thick brows"]),
    "Chubby": ("adj", ["chubby", "plump", "round-faced", "full-cheeked", "heavyset in the face"]),
    "Double_Chin": ("noun", ["a double chin", "a second chin", "extra fold under the chin", "a fleshy under-chin", "a sagging under-chin"]),
    "Eyeglasses": ("noun", ["eyeglasses", "glasses", "spectacles", "a pair of glasses", "reading glasses"]),
    "Goatee": ("noun", ["a goatee", "a goatee beard", "a small pointed beard", "a chin goatee", "a trimmed goatee"]),
    "Gray_Hair": ("noun", ["gray hair", "grey hair", "silver hair", "white-gray hair", "salt-and-pepper hair"]),
    "Heavy_Makeup": ("noun", ["heavy makeup", "a lot of makeup", "thick makeup", "bold heavy makeup", "heavily applied makeup"]),
    "High_Cheekbones": ("noun", ["high cheekbones", "prominent cheekbones", "sharp high cheekbones", "well-defined cheekbones", "raised cheekbones"]),
    "Male": ("noun", ["a male face", "the face of a man", "masculine features", "a man's face", "a masculine appearance"]),
    "Mouth_Slightly_Open": ("noun", ["a slightly open mouth", "lips parted slightly", "a mouth slightly open", "barely parted lips", "a softly open mouth"]),
    "Mustache": ("noun", ["a mustache", "a moustache", "hair on the upper lip", "a thick mustache", "a trimmed mustache"]),
    "Narrow_Eyes": ("noun", ["narrow eyes", "squinted eyes", "thin eyes", "slightly closed eyes", "narrowed eyes"]),
    "No_Beard": ("adj", ["clean-shaven", "without a beard", "with a beardless face", "smooth-shaven", "free of facial hair"]),
    "Oval_Face": ("noun", ["an oval face", "an oval-shaped face", "a long oval face", "a softly oval face", "an egg-shaped face"]),
    "Pale_Skin": ("noun", ["pale skin", "fair skin", "light-toned skin", "very pale skin", "porcelain-pale skin"]),
    "Pointy_Nose": ("noun", ["a pointy nose", "a sharp nose", "a pointed nose", "a narrow pointed nose", "a thin pointy nose"]),
    "Receding_Hairline": ("noun", ["a receding hairline", "a hairline pulled back", "thinning hair at the temples", "a balding hairline", "a hairline receding at the front"]),
    "Rosy_Cheeks": ("noun", ["rosy cheeks", "pink cheeks", "flushed cheeks", "reddish rosy cheeks", "blushing cheeks"]),
    "Sideburns": ("noun", ["sideburns", "long sideburns", "hair down the sides of the face", "thick sideburns", "sideburns along the cheeks"]),
    "Smiling": ("adj", ["smiling", "with a smile", "grinning", "smiling broadly", "happily smiling"]),
    "Straight_Hair": ("noun", ["straight hair", "sleek straight hair", "perfectly straight hair", "smooth straight hair", "flat straight hair"]),
    "Wavy_Hair": ("noun", ["wavy hair", "curly wavy hair", "loosely wavy hair", "soft wavy hair", "gently waving hair"]),
    "Wearing_Earrings": ("noun", ["earrings", "ear jewelry", "studs in the ears", "dangling earrings", "a pair of earrings"]),
    "Wearing_Hat": ("noun", ["a hat", "a cap on the head", "headwear", "a brimmed hat", "a hat on the head"]),
    "Wearing_Lipstick": ("noun", ["lipstick", "colored lipstick", "lip color", "bright lipstick", "bold lipstick"]),
    "Wearing_Necklace": ("noun", ["a necklace", "a chain around the neck", "neck jewelry", "a pendant necklace", "a beaded necklace"]),
    "Wearing_Necktie": ("noun", ["a necktie", "a tie", "a knotted tie", "a formal necktie", "a tie around the collar"]),
    "Young": ("adj", ["young", "youthful", "young-looking", "youthful-faced", "in their youth"]),
}

def build_prompts_for_attribute(name):
    # Cross sentence frames × synonym phrases → de-duplicated prompt list for one attribute.
    # The frame kind (noun/adj) picks FRAMES vs PREDICATIVE_FRAMES; this spread IS the subspace.
    kind, phrases = ATTR_PHRASES[name]
    frames = PREDICATIVE_FRAMES if kind == "adj" else FRAMES
    out, seen = [], set()
    for phrase in phrases:
        for frame in frames:
            text = frame.format(phrase=phrase)
            if text not in seen:
                seen.add(text); out.append(text)
    return out

assert set(ATTR_PHRASES) == set(ATTRIBUTE_NAMES), "prompt phrases must cover exactly the 40 attributes"

In [ ]:
@torch.no_grad()
def load_or_extract_prompt_bank(force=False):
    # Load-or-build the [40, n, 512] prompt bank: bank[j] is attribute j's prompt stack,
    # each row L2-normalized. Per-attr counts differ, so stacks are PADDED to a common n by
    # repeating the first prompt (SVD-safe: a duplicate row adds no new span direction).
    path = ARTIFACTS_DIR / "clip_attr_prompt_bank.pt"
    if path.exists() and not force:
        bank = torch.load(path, weights_only=True); print(f"[OK] loaded prompt bank {tuple(bank.shape)} (cache)")
        return bank
    from transformers import CLIPModel, AutoTokenizer
    model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(CLIP_MODEL_NAME)
    per_attr = [build_prompts_for_attribute(n) for n in ATTRIBUTE_NAMES]
    n = max(len(p) for p in per_attr)
    print(f"  prompts/attr: min={min(len(p) for p in per_attr)}, max={n} (padded to {n})")
    bank = torch.empty(40, n, FEATURE_DIM, dtype=torch.float32)
    for j, prompts in enumerate(per_attr):
        padded = prompts + [prompts[0]] * (n - len(prompts))
        toks = tok(padded, padding=True, return_tensors="pt").to(DEVICE)
        out = model.text_model(**toks)
        bank[j] = torch.nn.functional.normalize(model.text_projection(out.pooler_output), p=2, dim=1).cpu().float()
        print(f"  encoded {j+1}/40", end="\r")
    print()
    torch.save(bank, path); print(f"  saved {path} {tuple(bank.shape)}")
    return bank

prompt_bank = load_or_extract_prompt_bank()

---
## 7. Modality-gap analysis (the fact that motivates everything)

Before designing fusion, we verify the modality gap on *our* tensors. We compare three cosine
distributions — image↔image, text↔text, image↔text — and the angle between the two cone
centroids `μ_img`, `μ_txt`. If images and text shared one region, cross-modal cosines would
overlap the within-modal ones. They don't: the cross-modal mass sits far lower, and `μ_img`,
`μ_txt` are nearly orthogonal. **Consequence:** naïvely adding `t(a)` to `v_ref` mostly injects
the constant `μ_txt` offset (no attribute signal) — which is exactly what Tier-0+ FIX 1 removes.


In [ ]:
import matplotlib.pyplot as plt

def modality_gap_report(image_features, attr_text_features, n_sample=3000):
    # Quantify CLIP's modality gap: within-image vs within-text vs cross-modal cosines, and
    # the angle between the cone centroids. The separation is what FIX 1 (centering) corrects.
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(image_features.shape[0], generator=g)[:n_sample]
    imgs = image_features[idx]
    ii = (imgs @ imgs.T)[torch.triu(torch.ones(n_sample, n_sample), 1) == 1]
    tt = (attr_text_features @ attr_text_features.T)
    tt = tt[torch.triu(torch.ones(40, 40), 1) == 1]
    it = (imgs @ attr_text_features.T).flatten()
    mu_img = torch.nn.functional.normalize(image_features.mean(0), dim=0)
    mu_txt = torch.nn.functional.normalize(attr_text_features.mean(0), dim=0)
    gap_deg = torch.rad2deg(torch.arccos((mu_img @ mu_txt).clamp(-1, 1))).item()

    print(f"mean cos  image-image = {ii.mean():.3f} | text-text = {tt.mean():.3f} | image-text = {it.mean():.3f}")
    print(f"angle between cone centroids ∠(μ_img, μ_txt) = {gap_deg:.1f}°  (≈90° ⇒ disjoint cones)")
    plt.figure(figsize=(7, 3.5))
    for vals, lbl in [(ii, "image–image"), (tt, "text–text"), (it, "image–text")]:
        plt.hist(vals.numpy(), bins=60, alpha=0.5, density=True, label=lbl)
    plt.xlabel("cosine similarity"); plt.ylabel("density"); plt.legend(); plt.title("CLIP modality gap")
    plt.tight_layout(); plt.show()

modality_gap_report(image_features, attr_text_features)

---
## 8. Shared geometry on the hypersphere

CLIP features live on the unit sphere `S^{d-1}`, so the "right" arithmetic is Riemannian, not
Euclidean. Tier-1, Tier-2a and Tier-2b all reuse these primitives (defined **once** here):

- **Log / Exp maps** (GDE App. A): move between the sphere and the flat tangent plane at a base
  point `μ`. `log_μ(x) = θ·(x − cosθ·μ)/sinθ`, `θ = arccos(xᵀμ)`; `exp_μ(v) = cos‖v‖·μ + sin‖v‖·v̂`.
- **`align_rotation(a,b)`** — minimal rotation `H` sending `a → b` inside `span{a,b}`, identity
  elsewhere (CLAY's modality-gap rotation: align the visual mean to the text mean without
  disturbing intra-DB relations).
- **`build_subspace(stack, k)`** — CLAY's manifold-aware subspace: `μ_c =` normalized mean,
  SVD of `log_{μ_c}(stack)`, keep the top-k right singular vectors `V_k`.
- **`intrinsic_mean`** (Karcher mean) and **`tangent_mean`** (GDE primitive direction) for Tier-2b.


In [ ]:
import torch.nn.functional as F

def log_map(mu, X, eps=1e-6):
    # Log map — project sphere points X [m,d] onto the tangent plane at unit μ [d].
    # log_μ(x) = θ·(x − cosθ·μ)/sinθ. Points at μ (θ≈0) map to 0 (eps guards 0/0).
    dots = (X @ mu).clamp(-1.0, 1.0)
    theta = torch.arccos(dots)
    tangent = X - dots.unsqueeze(1) * mu
    norms = tangent.norm(dim=1, keepdim=True)
    scale = torch.where(norms > eps, theta.unsqueeze(1) / norms, torch.zeros_like(norms))
    return tangent * scale

def exp_map(mu, V, eps=1e-6):
    # Exp map — lift tangent vectors V [m,d] at μ back onto the sphere. Zero tangent → μ.
    norms = V.norm(dim=1, keepdim=True)
    v_hat = torch.where(norms > eps, V / norms, torch.zeros_like(V))
    return torch.cos(norms) * mu + torch.sin(norms) * v_hat

def align_rotation(a, b, eps=1e-6):
    # Minimal rotation H sending unit a → unit b, identity on span{a,b}^⊥ (CLAY §3.2).
    # Closes the modality gap by rotating only within the 2-D plane of the two means.
    d = a.shape[0]; I = torch.eye(d, dtype=a.dtype)
    c = float(a @ b); u2 = b - c * a; s = u2.norm()
    if s < eps:                                   # (anti)parallel → nothing to rotate
        return I
    u2 = u2 / s
    P = torch.outer(a, a) + torch.outer(u2, u2)   # projector onto the plane
    R = c * torch.outer(a, a) + s * torch.outer(u2, a) - s * torch.outer(a, u2) + c * torch.outer(u2, u2)
    return I - P + R

def build_subspace(stack, k):
    # Manifold-aware textual subspace (CLAY §3.2): tangent point μ_c + top-k right singular
    # vectors V_k of log_{μ_c}(stack). k is clamped to the stack height.
    mu_c = F.normalize(stack.mean(dim=0), dim=0)
    _, _, Vh = torch.linalg.svd(log_map(mu_c, stack), full_matrices=False)
    k_eff = min(k, Vh.shape[0])
    return mu_c, Vh[:k_eff].T                     # (μ_c [d], V_k [d, k_eff])

def intrinsic_mean(X, n_iter=50, lr=1.0, eps=1e-7):
    # Karcher mean — the point on S^{d-1} minimising Σ d²(μ, x_i) (GDE Alg. 1).
    # Warm-started from the normalized Euclidean mean; converges in <20 iters on CelebA.
    mu = F.normalize(X.mean(dim=0), dim=0)
    for _ in range(n_iter):
        grad = log_map(mu, X).mean(dim=0)
        if grad.norm() < eps:
            break
        mu = F.normalize(exp_map(mu.unsqueeze(0), (lr * grad).unsqueeze(0)).squeeze(0), dim=0)
    return mu

def tangent_mean(mu, X):
    # Primitive direction for an attribute (GDE Prop. 1, Eq. 7): v_a = mean_i log_μ(x_i).
    # NOT normalized — the magnitude encodes average angular displacement (signal for GDE).
    return log_map(mu, X).mean(dim=0)

---
## 9. Tier 0 — vanilla latent-arithmetic baseline (the floor)

The simplest compositional retrieval: push the reference vector toward positive attributes and
away from negatives by plain vector addition, then rank by cosine.

$$q = \mathrm{normalize}\!\Big(v_\text{ref} + \alpha\big(\textstyle\sum_i t^{+}_i - \sum_j t^{-}_j\big)\Big),\qquad \text{score}(d)=\cos(q, v_d)$$

No manifold awareness, no subspaces, no learning. This produces the first real Recall@K numbers
and the floor every later tier must clear. `α` is the identity↔modification dial (an ablation knob).
*(As §7 shows, adding raw text vectors to an image vector is geometrically crude — Tier-0+ fixes
exactly this.)*


In [ ]:
def tier0_score(T_pos, T_neg, v_ref_idx, image_features, attr_text_features, alpha=1.0):
    # Tier-0 scorer — q = normalize(v_ref + α(Σt⁺ − Σt⁻)), ranked by cosine over the frozen DB.
    # Raw latent arithmetic; returns dataset indices best-first with the source excluded (§5).
    q = image_features[v_ref_idx].clone()
    delta = torch.zeros_like(q)
    for name in T_pos: delta += attr_text_features[ATTR_TO_IDX[name]]
    for name in T_neg: delta -= attr_text_features[ATTR_TO_IDX[name]]
    q = F.normalize(q + alpha * delta, p=2, dim=0)
    scores = image_features @ q
    scores[v_ref_idx] = float("-inf")
    return torch.argsort(scores, descending=True).tolist()

def tier0_make(query_str, image_features, attr_text_features, alpha=1.0):
    # Curry one parsed query into the get_ranking(src_idx) callback the eval engine expects.
    T_pos, T_neg = parse_query(query_str)
    return lambda s: tier0_score(T_pos, T_neg, s, image_features, attr_text_features, alpha)

def evaluate_tier0(alpha=1.0, ks=KS):
    return evaluate_all(gt_list, lambda q: tier0_make(q, image_features, attr_text_features, alpha), ks)

res_tier0 = register("Tier-0 (α=1)", evaluate_tier0(alpha=1.0))
show(res_tier0, "Tier-0 — vanilla latent arithmetic (α=1.0)")

---
## 10. Tier 0+ — vanilla arithmetic with **correct** CLIP geometry

Same family (no SVD, no learning), but we fix three purely *geometric* mistakes the naïve
baseline makes — each a mean-subtraction or renormalization, so it stays "vanilla arithmetic,
done right" and leaves Tier-1/2 their full headroom. Each is an independent toggle (ablation).

- **FIX 1 — modality-gap centering** (Liang et al., 2022). Subtract the common text offset
  `μ_txt` from each attribute direction and do the image-side arithmetic in the `μ_img`-centered
  frame. `α` then buys *attribute direction*, not constant "text-ness". **This is the dominant lever.**
- **FIX 2 — delta normalization.** `‖Σt⁺ − Σt⁻‖` grows with the *number* of constraints, so a
  fixed `α` secretly means a different angular step per query. Normalize the delta so `α` is one
  honest angle on the sphere for all queries.
- **FIX 3 — prompt ensembling** (Radford et al., 2021). Average the `n` paraphrases of the prompt
  bank (then renormalize): the shared concept adds coherently while per-phrasing noise cancels.

$$\hat t(a)=\mathrm{normalize}\big(\mathrm{normalize}(\textstyle\frac1n\sum_k \text{bank}[a,k])-\mu_\text{txt}\big),\quad d=\mathrm{normalize}\big(\textstyle\sum\hat t^{+}-\sum\hat t^{-}\big),\quad q=\mathrm{normalize}\big((v_\text{ref}-\mu_\text{img})+\alpha d+\mu_\text{img}\big)$$

**Order is load-bearing:** ensemble (FIX 3) *then* center (FIX 1) — averaging paraphrases does
not remove the common-mode `μ_txt`; only the explicit subtraction does.


In [ ]:
def build_attr_directions(image_features, prompt_bank, attr_text_features, use_prompt_bank=True, center=True):
    # Per-attribute unit directions: FIX 3 (ensemble paraphrases) THEN FIX 1 (strip μ_txt).
    # Returns (dirs [40,512] unit, μ_img) so the caller centers the image side with the SAME μ_img.
    mu_img = image_features.mean(dim=0)
    if use_prompt_bank:
        attr = F.normalize(prompt_bank.mean(dim=1), p=2, dim=1)   # FIX 3: centroid of paraphrases
    else:
        attr = attr_text_features.clone()
    if center:
        attr = F.normalize(attr - attr.mean(dim=0), p=2, dim=1)   # FIX 1: remove common text offset
    return attr, mu_img

def tier0e_score(T_pos, T_neg, v_ref_idx, image_features, attr_dirs, mu_img, alpha=1.0, norm_delta=True, center=True):
    # Tier-0 ENHANCED scorer — latent arithmetic in CORRECT CLIP geometry.
    # q = normalize((v_ref − μ_img) + α·d + μ_img),  d = normalize(Σt̂⁺ − Σt̂⁻). Source excluded.
    v_ref = image_features[v_ref_idx].clone()
    delta = torch.zeros_like(v_ref)
    for name in T_pos: delta += attr_dirs[ATTR_TO_IDX[name]]
    for name in T_neg: delta -= attr_dirs[ATTR_TO_IDX[name]]
    if norm_delta and delta.norm() > 1e-8:                        # FIX 2: α becomes one honest angle
        delta = F.normalize(delta, p=2, dim=0)
    q = (v_ref - mu_img) + alpha * delta + mu_img if center else v_ref + alpha * delta
    q = F.normalize(q, p=2, dim=0)
    scores = image_features @ q
    scores[v_ref_idx] = float("-inf")
    return torch.argsort(scores, descending=True).tolist()

def evaluate_tier0e(alpha=1.0, use_prompt_bank=True, center=True, norm_delta=True, ks=KS):
    attr_dirs, mu_img = build_attr_directions(image_features, prompt_bank, attr_text_features,
                                              use_prompt_bank=use_prompt_bank, center=center)
    def make(q):
        T_pos, T_neg = parse_query(q)
        return lambda s: tier0e_score(T_pos, T_neg, s, image_features, attr_dirs, mu_img,
                                      alpha=alpha, norm_delta=norm_delta, center=center)
    return evaluate_all(gt_list, make, ks)

# Ablation: naive-equivalent → each fix alone → all three. (FIX 1 centering is the big win.)
abl = pd.DataFrame([
    {"config": tag, **mean_metrics(evaluate_tier0e(use_prompt_bank=b, center=c, norm_delta=nd))}
    for tag, b, c, nd in [("naive", False, False, False), ("fix3_bank", True, False, False),
                          ("fix1_center", False, True, False), ("fix2_normdelta", False, False, True),
                          ("all_fixes", True, True, True)]
]).round(4)
print("Tier-0+ fix ablation (α=1.0):");
try:
    from IPython.display import display; display(abl)
except Exception:
    print(abl.to_string(index=False))

res_tier0e = register("Tier-0+ (centering)", evaluate_tier0e(use_prompt_bank=False, center=True, norm_delta=False))
register("Tier-0+ (all fixes)", evaluate_tier0e(use_prompt_bank=True, center=True, norm_delta=True))

---
## 11. Tier 1 — CLAY reproduction (the method-to-beat)

Faithful pure CLAY (Lim et al., 2026): build **one** manifold-aware subspace from *all* the
condition prompts stacked together, project the frozen DB into it, and score by cosine *inside*
the subspace.

$$T_c=\text{stack of all condition prompts},\quad \mu_c=\mathrm{normalize}(\overline{T_c}),\quad
\log_{\mu_c}(T_c)=U\Sigma V^\top,\ V_k=\text{top-}k$$
$$m_\text{CLAY}(v)=V_k^\top\,\log_{\mu_c}\!\big(H\,v\big),\qquad \text{csim}=\cos\!\big(m_\text{CLAY}(v_\text{ref}),\,m_\text{CLAY}(v_d)\big)$$

This is the **naïve pre-SVD stack** the project attacks: positives and negatives are merged into
*undifferentiated* condition axes — CLAY has **no native negation**. It is a *focus/preserve*
operation, not a *modification*, so it is expected to lag on this modification-heavy, negation-bearing
benchmark. `H` aligns the visual mean to the text mean (modality gap). We ablate `k` and `H`.


In [ ]:
def _stack_condition_prompts(T_pos, T_neg, prompt_bank):
    # CLAY's naïve pre-SVD stack: every condition's UNPADDED prompts in one matrix [m,d].
    # The bank is padded with duplicate rows; we slice to the true per-attr count so the
    # duplicates don't skew μ_c / the singular values. Positives and negatives are merged.
    rows = []
    for name in T_pos + T_neg:
        j = ATTRIBUTE_NAMES.index(name)
        rows.append(prompt_bank[j, :len(build_prompts_for_attribute(name))])
    return torch.cat(rows, dim=0)

def _project_db(image_features, mu_c, V_k, use_rotation=True):
    # Project the frozen DB into the conditional subspace ONCE per query:
    # D = normalize_rows( log_{μ_c}(H·v) @ V_k ). H aligns the visual cone to the text mean.
    V = image_features
    if use_rotation:
        mu_vd = F.normalize(image_features.mean(dim=0), dim=0)
        V = V @ align_rotation(mu_vd, mu_c).T
    return F.normalize(log_map(mu_c, V) @ V_k, dim=1, eps=1e-12)

def clay_make(query_str, image_features, prompt_bank, k=50, use_rotation=True):
    # Build get_ranking, precomputing the subspace + projected DB once per query (CONTRACT §7).
    T_pos, T_neg = parse_query(query_str)
    mu_c, V_k = build_subspace(_stack_condition_prompts(T_pos, T_neg, prompt_bank), k)
    D = _project_db(image_features, mu_c, V_k, use_rotation)
    def get_ranking(src):
        scores = D @ D[src]
        scores[src] = float("-inf")
        return torch.argsort(scores, descending=True).tolist()
    return get_ranking

def evaluate_tier1(k=50, use_rotation=True, ks=KS):
    return evaluate_all(gt_list, lambda q: clay_make(q, image_features, prompt_bank, k, use_rotation), ks)

# k / rotation ablation (k is monotonic and plateaus; H helps faithful CLAY).
clay_abl = pd.DataFrame([
    {"config": f"k={k}, {'rotH' if r else 'norot'}", **mean_metrics(evaluate_tier1(k=k, use_rotation=r))}
    for k, r in [(5, True), (10, True), (20, True), (50, True), (50, False)]
]).round(4)
print("Tier-1 CLAY ablation:")
try:
    from IPython.display import display; display(clay_abl)
except Exception:
    print(clay_abl.to_string(index=False))

res_tier1 = register("Tier-1 CLAY (k=50)", evaluate_tier1(k=50, use_rotation=True))
show(res_tier1, "Tier-1 — CLAY (k=50, rotation H)")

## Overview of Tier-2 methods

Tier-2 is the critical innovation layer: a systematic attack on negation in compositional retrieval.
CLIP cannot represent "not X" as a point, so we represent *each polarity as a subspace*. Three
complementary approaches:

- **Tier-2a (Asymmetric Conditional Subspaces):** Separate positive and negative subspaces, score
  them asymmetrically. Positive subspaces use CLAY's manifold-aware conditional similarity. Negative
  subspaces incur a *penalty* — energy inside means "looks like the thing we want absent". Identity
  anchor ensures reference identity is preserved.

- **Tier-2b (Visual-Prototype Composition):** Mine attribute directions directly from *train images*
  (no modality gap, no text phrasing noise), compose geodesically on the sphere. Negation is
  **orthogonal rejection** in the query's tangent space — delete the attribute axis entirely, not
  "add anti-X". Ablation: LDE shows what the spherical geometry buys over flat arithmetic.

- **Tier-2c (Visual-Subspace Negation):** Synthesizes 2a and 2b — keeps 2b's image-space query
  rejection mechanism, swaps 2b's single direction for 2a's k-dim subspace mined from train images
  at the global mean. The nesting property (k=1 ≈ Tier-2b) makes the k_neg sweep a clean ablation:
  does extra subspace spread carry signal beyond a single clean axis?

All three are training-free, no learned parameters, CLIP stays frozen throughout.


## 12. Tier-2a — Asymmetric Conditional Subspaces
Tier-2a fixes the failure every earlier tier shares: negation collapses, because CLIP cannot
represent "not X" as a *point* on the sphere. We represent each polarity as a **subspace** and
score asymmetrically:

$$\text{score}(s,d)=\underbrace{\cos(v_s,v_d)}_{\text{identity anchor}}+\underbrace{\operatorname*{mean}_{a\in T^{+}}\cos_{S^{+}_a}(v_d,v_s)}_{\text{positive focus (CLAY-style)}}-\underbrace{\lambda\!\!\sum_{b\in T^{-}}\big\lVert \operatorname{proj}_{S^{-}_b}(v_d)\big\rVert}_{\text{asymmetric negation penalty}}$$

- **Per-condition subspaces** (one SVD *per attribute*, PoS-grounded philosophy) so no attribute
  with more prompts dominates — vs. CLAY's single merged stack.
- **Negation as a subspace penalty:** energy of `v_d` inside a *negative* subspace means "looks
  like the thing we want absent", so it is pushed down. "Not Male" = "low energy in the Male
  subspace" — a whole region, not a single anti-Male point.
- **Identity anchor** `cos(v_s, v_d)`: each query is "an image *like the reference*, with +X,
  without −X". Removing the anchor reproduces CLAY's sub-Tier-0 behavior (ablation below).

This subspace view of negation follows **SpaceVLM** (Kazemi Ranjbar et al., 2025) — *"A but not N"*
as a region near A and far from N, generalized here from text points to per-condition subspaces; the
per-condition split follows **PoS-grounded subspaces** (Oldfield et al., 2023). That a steerable
negation direction exists in CLIP at all is independently corroborated by Sammani et al. (2026).


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Tier2aConfig:
    # Tier-2a hyperparameters (bundled so call sites stay short). Defaults = recommended start.
    k_pos: int = 20; k_neg: int = 20         # +/- subspace dimensionality
    lam: float = 0.1                          # negation penalty weight (mild 0.05–0.3 band)
    use_rotation: bool = True                 # modality-gap rotation H
    per_condition: bool = True                # one subspace per attribute (else stacked per polarity)
    identity_anchor: bool = True              # add the reference-identity cosine

def _attr_prompt_stack(name, prompt_bank):
    # One attribute's UNPADDED prompt stack [n_j, d] (strip the bank's duplicate padding).
    return prompt_bank[ATTRIBUTE_NAMES.index(name), :len(build_prompts_for_attribute(name))]

def _polarity_groups(names, prompt_bank, per_condition):
    # Group a polarity's prompts into the stacks each gets a subspace from: one per attribute
    # (per_condition) or one merged stack (CLAY-style). Empty names → no groups (degenerate case).
    stacks = [_attr_prompt_stack(n, prompt_bank) for n in names]
    if per_condition or not stacks:
        return stacks
    return [torch.cat(stacks, dim=0)]

def _project_db_into_subspace(stack, image_features, k, use_rotation):
    # Tangent coords of every DB image in the stack's subspace: log_{μ_c}(H·v) @ V_k → [N, k_eff].
    # Per-subspace H. Returns RAW coords; caller row-normalizes (S⁺ cosine) or takes the norm (S⁻ energy).
    mu_c, V_k = build_subspace(stack, k)
    V = image_features
    if use_rotation:
        V = V @ align_rotation(F.normalize(image_features.mean(dim=0), dim=0), mu_c).T
    return log_map(mu_c, V) @ V_k

def _precompute_S(T_pos, T_neg, image_features, prompt_bank, cfg):
    # Build the per-query, source-independent pieces ONCE (the expensive part, reused per source):
    # row-normalized D⁺ per positive group, and one summed S⁻ energy-penalty vector.
    pos_db = [F.normalize(_project_db_into_subspace(g, image_features, cfg.k_pos, cfg.use_rotation), dim=1, eps=1e-12)
              for g in _polarity_groups(T_pos, prompt_bank, cfg.per_condition)]
    neg_groups = _polarity_groups(T_neg, prompt_bank, cfg.per_condition)
    neg_penalty = None
    if neg_groups:
        neg_penalty = torch.zeros(image_features.shape[0])
        for g in neg_groups:
            neg_penalty = neg_penalty + _project_db_into_subspace(g, image_features, cfg.k_neg, cfg.use_rotation).norm(dim=1)
    return pos_db, neg_penalty

def _combine_S(image_features, cols, pos_db, neg_penalty, cfg):
    # Composite Tier-2a score for a batch of reference columns → [N, len(cols)] (the formula, once).
    scores = torch.zeros(image_features.shape[0], len(cols))
    if cfg.identity_anchor:
        scores = scores + image_features @ image_features[cols].T
    if pos_db:
        scores = scores + sum(D @ D[cols].T for D in pos_db) / len(pos_db)
    if neg_penalty is not None:
        scores = scores - cfg.lam * neg_penalty.unsqueeze(1)
    return scores

def tracks_make(query_str, image_features, prompt_bank, cfg=Tier2aConfig()):
    # get_ranking callback for one query (CONTRACT §7); subspaces precomputed once.
    T_pos, T_neg = parse_query(query_str)
    pos_db, neg_penalty = _precompute_S(T_pos, T_neg, image_features, prompt_bank, cfg)
    def get_ranking(src):
        scores = _combine_S(image_features, torch.tensor([src]), pos_db, neg_penalty, cfg)[:, 0]
        scores[src] = float("-inf")
        return torch.argsort(scores, descending=True).tolist()
    return get_ranking

def evaluate_tracks(cfg=Tier2aConfig(), ks=KS):
    # Batched driver: rank all of a query's sources at once (GEMM, not thousands of mat-vecs).
    results, top_k = {}, max(ks)
    for entry in gt_list:
        T_pos, T_neg = parse_query(entry["query"])
        pos_db, neg_penalty = _precompute_S(T_pos, T_neg, image_features, prompt_bank, cfg)
        srcs = [int(s) for s in entry["ground_truth"]]
        rankings = {}
        for start in range(0, len(srcs), 512):
            cols = torch.tensor(srcs[start:start + 512])
            scores = _combine_S(image_features, cols, pos_db, neg_penalty, cfg)
            scores[cols, torch.arange(len(cols))] = float("-inf")
            top = torch.topk(scores, top_k, dim=0).indices
            for j, s in enumerate(cols.tolist()):
                rankings[s] = top[:, j].tolist()
        results[entry["query"]] = evaluate_query(entry["ground_truth"], rankings.__getitem__, ks)
    return results

# Ablation: the identity anchor is decisive; k monotonic; per-condition > stacked; λ mild.
s_abl = pd.DataFrame([
    {"config": tag, **mean_metrics(evaluate_tracks(cfg))}
    for tag, cfg in [("default (k20, anchor)", Tier2aConfig()),
                     ("no anchor", Tier2aConfig(identity_anchor=False)),
                     ("k=50", Tier2aConfig(k_pos=50, k_neg=50)),
                     ("stacked", Tier2aConfig(per_condition=False)),
                     ("no rotation", Tier2aConfig(use_rotation=False))]
]).round(4)
print("Tier-2a Tier-2a ablation:")
try:
    from IPython.display import display; display(s_abl)
except Exception:
    print(s_abl.to_string(index=False))

res_tracks = register("Tier-2a Tier-2a", evaluate_tracks(Tier2aConfig(k_pos=50, k_neg=50)))
show(res_tracks, "Tier-2a — Tier-2a (asymmetric conditional subspaces, k=50)")

## 13. Tier-2b — Visual-Prototype Composition (GDE)
Tier-2b mines per-attribute **primitive directions** in *image* space (no modality gap, no
phrasing noise; concept directions in the lineage of CAVs, Kim et al., 2018) and composes the
query geodesically. Crucially, directions are mined from the
**train** split and ranking is on the **test** split — no leakage (GDE mines directions from
train and evaluates on test for exactly this reason).

**Mining (train).** Karcher mean `μ` of all train images, then `v_a = tangent_mean(μ, {x : has a})`.

**Composition (test).**
$$q_\text{tan}=\log_\mu(v_\text{ref})+\!\!\sum_{a\in T^{+}}\!\! v_a;\quad
\text{for }b\in T^{-}:\ q_\text{tan}\mathrel{-}=(q_\text{tan}\!\cdot\!\hat v_b)\,\hat v_b;\quad
q=\exp_\mu(q_\text{tan})$$

Negation is **orthogonal rejection**, not subtraction: it removes the attribute axis entirely
("not red hair" → blonde/brown/black all acceptable), avoiding the "anti-X" overshoot
(Alhamoud et al., 2025; Oldfield et al., 2023). **Ablation:** *LDE* (Trager et al., 2023) — the
same pipeline with flat Euclidean arithmetic and subtraction-negation — isolates what the
spherical geometry buys.

> Needs train artifacts: `clip_image_features_train.pt` (load-or-build below) and the train
> attribute mask. If the train split isn't staged, this section will extract it (slow, GPU).


In [ ]:
def load_train_data():
    # Train-split image features (load-or-build) + the [N_train, 40] attribute mask.
    celeba_train = load_celeba("train")
    train_attrs = celeba_train.attr.to(torch.float32)
    train_feats = load_or_extract_image_features("train", n_expected=len(celeba_train))
    return train_feats, train_attrs

def mine_directions(train_features, train_attributes, n_iter=50):
    # μ = intrinsic_mean(all train images); v_a = tangent_mean(μ, has-a images), per attribute.
    # Attributes with no train example get a zero direction (logged), not a crash.
    print("Computing intrinsic mean of train corpus …")
    mu = intrinsic_mean(train_features, n_iter=n_iter)
    directions = torch.zeros(40, train_features.shape[1], dtype=train_features.dtype)
    for j, name in enumerate(ATTRIBUTE_NAMES):
        has_a = train_features[train_attributes[:, j] > 0.5]
        if has_a.shape[0] == 0:
            print(f"  [!] no train images with '{name}' — direction zeroed"); continue
        directions[j] = tangent_mean(mu, has_a)
    print("[OK] direction mining complete.")
    return mu, directions

def load_or_mine_directions(force=False):
    # Cache mined (μ, directions) to artifacts/visual_directions.pt so re-runs are instant.
    path = ARTIFACTS_DIR / "visual_directions.pt"
    if path.exists() and not force:
        ckpt = torch.load(path, weights_only=True); print(f"[OK] loaded mined directions (cache)")
        return ckpt["mu"], ckpt["directions"]
    tf, ta = load_train_data()
    mu, directions = mine_directions(tf, ta)
    torch.save({"mu": mu, "directions": directions}, path); print(f"  saved {path}")
    return mu, directions

def _compose_gde(v_ref, T_pos, T_neg, mu, directions, eps=1e-8):
    # GDE composition — tangent-space addition for +, orthogonal rejection for −, then Exp.
    q_tan = log_map(mu, v_ref.unsqueeze(0)).squeeze(0)
    for name in T_pos:
        q_tan = q_tan + directions[ATTR_TO_IDX[name]]
    for name in T_neg:
        v_a = directions[ATTR_TO_IDX[name]]
        if v_a.norm() < eps:
            continue
        v_hat = v_a / v_a.norm()
        q_tan = q_tan - (q_tan @ v_hat) * v_hat           # remove the attribute axis entirely
    return F.normalize(exp_map(mu.unsqueeze(0), q_tan.unsqueeze(0)).squeeze(0), dim=0)

def _compose_lde(v_ref, T_pos, T_neg, directions):
    # LDE ablation — flat Euclidean arithmetic, subtraction-negation (Trager et al. 2023).
    q = v_ref.clone()
    for name in T_pos: q = q + directions[ATTR_TO_IDX[name]]
    for name in T_neg: q = q - directions[ATTR_TO_IDX[name]]
    return F.normalize(q, dim=0)

def trackv_make(query_str, image_features, mu, directions, use_gde=True):
    # get_ranking callback for one query; the query vector is built once, per-source = one dot product.
    T_pos, T_neg = parse_query(query_str)
    def get_ranking(src):
        v_ref = image_features[src]
        q = _compose_gde(v_ref, T_pos, T_neg, mu, directions) if use_gde else _compose_lde(v_ref, T_pos, T_neg, directions)
        scores = image_features @ q
        scores[src] = float("-inf")
        return torch.argsort(scores, descending=True).tolist()
    return get_ranking

def evaluate_trackv(use_gde=True, ks=KS):
    mu, directions = load_or_mine_directions()
    return evaluate_all(gt_list, lambda q: trackv_make(q, image_features, mu, directions, use_gde), ks)

res_trackv = register("Tier-2a Tier-2b (GDE)", evaluate_trackv(use_gde=True))
register("Tier-2a Tier-2b (LDE abl.)", evaluate_trackv(use_gde=False))
show(res_trackv, "Tier-2a — Tier-2b (GDE: geodesic + rejection negation)")

## 14. Tier-2c — Visual-Subspace Negation
Tier-2a and Tier-2b each fix one half of negation and break the other half:

| | negation is… | acts on the… | mined from… | weakness it carries |
|---|---|---|---|---|
| **Tier-2b** | a single direction `v̂_X`, rejected | **query** tangent vector | train *images* with X | one axis can't capture how X *varies* across faces |
| **Tier-2a** | a `k`-dim subspace, penalised | **DB** score | *text* prompts of X | the subspace is text-derived — it must cross the modality gap to identify male *images* |

**Tier-2c keeps the better half of each.** It inherits V's negation *mechanism* — orthogonal
rejection on the **query**, entirely in image space (no modality gap, no rotation `H`) — but
replaces V's single direction with S's richer **`k`-dim subspace**, mined by SVD from the **train
images that actually have X** (not from text). "Not Male" becomes *"delete the whole visual Male
region from the query"* — not "delete one average-Male axis" (V), and not "penalise text-Male
energy" (S). The positive side and the cosine scoring are inherited from Tier-2b unchanged.

**Mining (train, per negated attribute `b`).** Log-map the has-`b` images at the **global** mean
`μ` — the *same* tangent point the query lives in (load-bearing: a subspace built at a *local*
mean lives in a different tangent plane, so the rejection would be geometrically meaningless) —
then keep the top-`k` right singular vectors:
$$L_b=\log_\mu(X_b),\qquad Q_b=\operatorname{top-}k\ \text{right singular vectors of } L_b\quad(\perp\mu).$$

**Composition (test).** Add positives, reject the **union** of the negated subspaces from the
query tangent vector, exp back to the sphere:
$$q_\text{tan}=\log_\mu(v_\text{ref})+\!\!\sum_{a\in T^{+}}\!\! v_a;\qquad
Q_\text{all}=\mathrm{qr}\big([\,Q_b\,]_{b\in T^{-}}\big);\qquad
q_\text{tan}\mathrel{-}=Q_\text{all}Q_\text{all}^{\top}q_\text{tan};\qquad q=\exp_\mu(q_\text{tan}).$$

**Nesting — the clean ablation.** At `k_neg = 1`, `Q_b` is the top singular direction of the
log-mapped has-`b` images, which is ≈ `v̂_b` (their shared component), so **Tier-2c ≈ Tier-2b**.
The `k_neg` sweep therefore isolates *exactly* what extra subspace dimensions buy over V's single
direction. We also ablate **where** we reject: `query` (headline) vs `db` (project the whole DB
into the complement — the visual analogue of NCS; DB rows are **not** renormalised, so the
suppressed norm is used correctly by the cosine). No label leakage — every subspace is mined from
the **train** split, scored on **test**, exactly as Tier-2b.


In [ ]:
K_CACHE_NEG = 50   # max negative-subspace width mined once; the k_neg sweep just slices [:k] from it.

def build_visual_neg_subspace(X_b, mu, k):
    # Visual negative subspace: top-k right singular vectors of Log_μ(X_b) at the GLOBAL μ.
    # We eigendecompose the [d,d] Gram LᵀL (= SVD's V) rather than SVD the [m,d] L — identical
    # basis, no [m,d] U materialised, scales to m≫d. Global μ (NOT a local mean): the rejected
    # subspace must live in the SAME tangent plane as the query tangent vector.
    L = log_map(mu, X_b)
    _, evecs = torch.linalg.eigh(L.T @ L)                # ascending eigenpairs
    k_eff = min(k, evecs.shape[1])
    return evecs[:, -k_eff:].flip(1).contiguous()        # [d, k_eff], descending importance

def load_or_mine_neg_subspaces(force=False, k=K_CACHE_NEG):
    # Load-or-build the [40, d, k] negative visual subspaces (mine once from train; μ shared with Tier-2b).
    path = ARTIFACTS_DIR / "visual_neg_subspaces.pt"
    if path.exists() and not force:
        sub = torch.load(path, weights_only=True); print(f"[OK] loaded negative visual subspaces {tuple(sub.shape)} (cache)")
        return sub
    mu, _ = load_or_mine_directions()                    # global tangent point μ (from Tier-2b's cache)
    tf, ta = load_train_data()
    print("Mining negative visual subspaces (40 attrs) …")
    sub = torch.zeros(40, tf.shape[1], k, dtype=tf.dtype)
    for j, name in enumerate(ATTRIBUTE_NAMES):
        X_b = tf[ta[:, j] > 0.5]                          # train images that HAVE attribute j
        if X_b.shape[0] == 0:
            print(f"  [!] no train images with '{name}' — subspace zeroed"); continue
        Q = build_visual_neg_subspace(X_b, mu, k)
        sub[j, :, :Q.shape[1]] = Q
    torch.save(sub, path); print(f"  saved {path} {tuple(sub.shape)}")
    return sub

def build_union_basis(subspaces, T_neg, k_neg):
    # Orthonormal basis of the union of the query's negative visual subspaces (QR-merged). None if empty.
    if not T_neg:
        return None
    k = min(k_neg, subspaces.shape[-1])
    W = torch.cat([subspaces[ATTR_TO_IDX[b]][:, :k] for b in T_neg], dim=1)
    return torch.linalg.qr(W)[0]                          # [d, k_total] orthonormal columns

def evaluate_svunion(mu, directions, subspaces, k_neg=10, reject_on="query", ks=KS):
    # Batched Tier-2c driver (rank all of a query's sources at once — GEMM, not thousands of mat-vecs).
    # reject_on="query" deletes the union subspace from each query tangent vector; "db" projects the
    # whole DB into the complement once per query and scores there (rows NOT renormalised).
    results, top_k = {}, max(ks)
    for entry in gt_list:
        T_pos, T_neg = parse_query(entry["query"])
        Q_all = build_union_basis(subspaces, T_neg, k_neg)
        base = image_features
        if reject_on == "db" and Q_all is not None:
            base = image_features - (image_features @ Q_all) @ Q_all.T   # complement; suppressed norm kept
        srcs = [int(s) for s in entry["ground_truth"]]
        rankings = {}
        for start in range(0, len(srcs), 512):
            cols = torch.tensor(srcs[start:start + 512])
            q_tan = log_map(mu, image_features[cols])                    # [m, d]
            for name in T_pos:
                q_tan = q_tan + directions[ATTR_TO_IDX[name]]
            if reject_on == "query" and Q_all is not None:
                q_tan = q_tan - (q_tan @ Q_all) @ Q_all.T                # reject union span per row
            Q = F.normalize(exp_map(mu, q_tan), dim=1)                   # [m, d] unit queries
            scores = base @ Q.T                                          # [N, m]
            scores[cols, torch.arange(len(cols))] = float("-inf")        # source never ranks itself
            top = torch.topk(scores, top_k, dim=0).indices
            for j, s in enumerate(cols.tolist()):
                rankings[s] = top[:, j].tolist()
        results[entry["query"]] = evaluate_query(entry["ground_truth"], rankings.__getitem__, ks)
    return results

mu, directions = load_or_mine_directions()
neg_subspaces  = load_or_mine_neg_subspaces()

# k_neg sweep × reject side. k_neg=1 nests Tier-2b; growing k tests whether the visual subspace's
# spread carries signal beyond V's single direction — it plateaus through k=5, then over-rejects.
sv_abl = pd.DataFrame([
    {"config": f"{side}, k_neg={k}", **mean_metrics(evaluate_svunion(mu, directions, neg_subspaces, k_neg=k, reject_on=side))}
    for side in ("query", "db") for k in (1, 5, 10, 20)
]).round(4)
print("Tier-2a Tier-2aV-union ablation (k_neg=1 ≈ Tier-2b):")
try:
    from IPython.display import display; display(sv_abl)
except Exception:
    print(sv_abl.to_string(index=False))

res_svunion = register("Tier-2a Tier-2c (k=5)", evaluate_svunion(mu, directions, neg_subspaces, k_neg=5, reject_on="query"))
show(res_svunion, "Tier-2a — Tier-2aV-union (visual-subspace query rejection, k_neg=5)")


---
## 15. Results & discussion

### 15.1 Master comparison

Every tier registered its macro-averaged metrics; the table and chart below are generated live
from the runs above (so they always match this notebook's CLIP build and artifacts).


In [1]:
master = pd.DataFrame(ALL_RESULTS).T[[f"R@{k}" for k in KS] + [f"P@{k}" for k in KS]].round(4)
print("Master comparison (macro-averaged over 14 queries):")
try:
    from IPython.display import display; display(master)
except Exception:
    print(master.to_string())

ax = master[[f"R@{k}" for k in KS]].plot(kind="bar", figsize=(10, 4))
ax.set_ylabel("Recall@K (hit rate)"); ax.set_title("Recall@K by method")
ax.set_xticklabels(master.index, rotation=20, ha="right")
plt.tight_layout(); plt.show()

NameError: name 'pd' is not defined

### 15.2 Reading the numbers (what we see, and why)

- **Tier-0 is a genuine, honest floor.** Its low scores are a property of the *method × benchmark*,
  not a bug: the relaxed-Hamming ground truth demands targets that both satisfy the query **and**
  stay within Hamming-2 of the source, so "ignore the text" (α=0) and "drown the identity" (α=1)
  both fail — exactly the fusion problem the assignment poses.
- **Tier-0+ centering nearly doubles the floor at R@5** with one mean-subtraction — empirical
  confirmation that the modality gap (§7), not the prompts, was the dominant error in naïve arithmetic.
- **Faithful CLAY (Tier-1) lands *below* Tier-0.** That is the *expected, gradeable* result: CLAY is
  a focus/preserve *similarity* reframing with **no +/− arithmetic**, so it trails on modification
  queries — the precise naïve-stacking bottleneck our Tier-2a attacks.
- **Tier-2a beats CLAY** — the asymmetric subspaces + identity anchor lift it well above Tier-1; the
  ablation shows the anchor is the decisive piece.
- **Tier-2b (GDE) beats CLAY on every query and reaches near-parity with Tier-0**, winning on `+Male`
  and on the negation query `-Heavy_Makeup`: a clean *visual* axis removal beats subtracting a
  modality-gapped text vector.
- **Tier-2c confirms the ceiling — and this is the decisive result.** Our synthesis of S and V
  *nests* Tier-2b: at `k_neg = 1` its Recall equals V's to the third decimal (the top singular
  direction recovers V's mined axis). Growing `k_neg` does **not** help — R@5 plateaus through `k=5`
  then *declines* (`k=10, 20`) as the wider subspace over-rejects, stripping identity content along
  with the attribute. `reject_on="db"` tracks `query` to within noise (marginally stronger on pure
  negation). **The extra spread of how an attribute varies across faces carries no *net* retrieval
  signal over a single clean axis** — the best training-free negation we can build ties, but does not
  beat, the single-direction rejection.

### 15.3 The negation story → why we turn to training

`-Male & -Mustache` stays at **0.000 for every method here**, and Tier-2c — the strongest
training-free negation we could engineer — does not move it. This is not a tuning failure; it is
structural. All 27 sources are themselves `Male=1, Mustache=1`, and a valid target must additionally
sit within Hamming-2 of the source on the other 38 attributes — i.e. the rare "female twin" of a
*specific* male face. Deleting the Male/Mustache region from the query is **necessary but not
sufficient**: ranking those few twins above ~12k other valid-negation faces needs identity signal,
and the only identity signal available is `v_ref` itself — which, being male, pulls the *wrong* way.

This is the **identity-anchor vs negation tension**, and it is the empirical finding that ends the
training-free line. Subtraction (Tier-0, LDE) overshoots to "anti-X"; CLAY has no negation; and the
subspace penalty (S), the geodesic rejection (V), and its full subspace generalisation (Tier-2c)
all converge to the **same ceiling** — every one of them still below the corrected text-push of
Tier-0+. Resolving the tension requires *learning to separate identity from the negated attribute*
rather than hand-deleting a fixed axis, which no frozen-geometry operator can do. That is precisely
the role of a trained fusion module Φ (§16).

### 15.4 Qualitative retrieval grids (success & failure)

For a chosen `(query, source)`, we show the reference and the top-K retrieved images, framing each
result **green** if it is in the ground-truth set and **red** otherwise. We display from a CelebA
view with **no normalization** (the CLIP-normalized tensors are not human-viewable).


In [ ]:
# A separate CelebA view that yields raw PIL images for display (no CLIP normalization).
celeba_raw = datasets.CelebA(root=DATASET_ROOT, split="test", download=False, transform=None)

def show_retrieval(query_str, src_idx, get_ranking, gt_targets, k=5, title=""):
    # Reference + top-K retrieved; green frame = in ground truth, red = miss. Visual sanity check.
    ranking = get_ranking(src_idx)[:k]
    gt = set(int(t) for t in gt_targets)
    fig, axes = plt.subplots(1, k + 1, figsize=(2.0 * (k + 1), 2.4))
    axes[0].imshow(celeba_raw[src_idx][0]); axes[0].set_title(f"ref #{src_idx}", fontsize=9)
    axes[0].axis("off")
    for ax, idx in zip(axes[1:], ranking):
        ax.imshow(celeba_raw[idx][0])
        ok = idx in gt
        for sp in ax.spines.values():
            sp.set_color("limegreen" if ok else "red"); sp.set_linewidth(3)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_title(("✓ " if ok else "✗ ") + str(idx), fontsize=9)
    fig.suptitle(title or f"{query_str}  (source {src_idx})", fontsize=11)
    plt.tight_layout(); plt.show()

def first_source(query_str):
    # First source index of a query in the eval JSON (a convenient demo pick).
    e = next(e for e in gt_list if e["query"] == query_str)
    src = next(iter(e["ground_truth"]))
    return int(src), e["ground_truth"][src]

# Example: Tier-2b (GDE) on a positive query (success) and on a harder negation query.
mu, directions = load_or_mine_directions()
for q in ["+Smiling", "-Heavy_Makeup"]:
    if any(e["query"] == q for e in gt_list):
        s, tgts = first_source(q)
        gr = trackv_make(q, image_features, mu, directions, use_gde=True)
        show_retrieval(q, s, gr, tgts, k=5, title=f"Tier-2b (GDE) — {q}")

---
## 16. Conclusion: the training-free ceiling, and the pivot to a trained Φ

**What we built.** A single frozen-CLIP retrieval spine and an escalating ladder of *training-free*
methods, all scored through one verified harness: a vanilla baseline (Tier-0), its geometry-corrected
form (Tier-0+), a faithful CLAY reproduction (Tier-1, the method-to-beat), our two complementary
contributions (Tier-2a Tier-2a and Tier-2b), and their **synthesis, Tier-2c**, which fuses V's
image-space query rejection with S's multi-dimensional subspace.

**Our gradeable insight.** Negation is *not* subtraction. Modeling "−X" as a subspace penalty
(Tier-2a) or as orthogonal rejection in the tangent space (Tier-2b) is principled and beats CLAY,
with zero learned parameters — CLIP stays frozen throughout.

**The ceiling we hit — and why it sends us to training.** Tier-2c is the controlled experiment that
closes the training-free line. It *nests* Tier-2b at `k_neg=1`, and widening the visual subspace adds
no net retrieval signal — it plateaus, then over-rejects. So the subspace penalty, the single-axis
rejection, and the full subspace rejection all converge to the **same ceiling**, all below Tier-0+'s
corrected text-push. The residue is the **identity-anchor vs negation tension**, exposed sharply by
`-Male & -Mustache` (structurally 0.000 for every method): a pure-negation query needs the model to
*keep the reference identity while discarding the negated attribute that is part of that identity* —
two goals a frozen geometric operator cannot separate by hand-deleting an axis. **This is the result
that led us to a training-based approach.**

**The pivot: a trained fusion module Φ.** The natural next step is a lightweight Φ over
`[v_ref, t⁺…, t⁻…]` with role/sign embeddings (cross-attention / FiLM), trained with InfoNCE on
synthetic train-split queries to *learn* the identity-vs-attribute separation that Tier-2c showed is
unreachable training-free. Everything that notebook needs already exists here: the verified harness,
the frozen image/text database, the mined geometry, and the synthetic-query substrate. The
training-free ladder in this notebook is both our honest baseline and the diagnostic that defines
exactly what Φ must learn.

## References

**Backbone & dataset**
- Radford, Kim, Hallacy, Ramesh, Goh, et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision* (CLIP). ICML. — the frozen backbone (spec §3.3).
- Liu, Luo, Wang, Tang (2015). *Deep Learning Face Attributes in the Wild* (CelebA). ICCV. — the dataset.
- Liang, Zhang, Kwon, Yeung, Zou (2022). *Mind the Gap: Understanding the Modality Gap in Multi-modal Contrastive Learning*. NeurIPS. — motivates the modality-gap centering (§7, Tier-0+ FIX 1).

**Compositional structure & methods we build on**
- Berasi, Farina, Mancini, Ricci, Strisciuglio (2025). *Not Only Text: Exploring Compositionality of Visual Representations in Vision-Language Models* (GDE). CVPR. arXiv:2503.17142. — geodesic decomposability; basis of Tier-2a Tier-2b.
- Lim, Hyoseok, Park, Oh (2026). *CLAY: Conditional Visual Similarity Modulation in Vision-Language Embedding Space*. CVPR (KAIST). — the Tier-1 method-to-beat.
- Trager, Perera, Zancato, Achille, Bhatia, Soatto (2023). *Linear Spaces of Meanings: Compositional Structures in Vision-Language Models* (LDE). ICCV (AWS AI Labs). — the flat-arithmetic ablation for Tier-2b.
- Oldfield, Tzelepis, Panagakis, Nicolaou, Patras (2023). *Parts of Speech–Grounded Subspaces in Vision-Language Models* (PoS-Subspaces). NeurIPS. — per-role/per-condition subspaces; basis of Tier-2a Tier-2a.
- Kim, Wattenberg, Gilmer, Cai, Wexler, Viégas, Sayres (2018). *Interpretability Beyond Feature Attribution: Quantitative Testing with Concept Activation Vectors* (TCAV). ICML. — concept directions (CAVs), the interpretability lineage of Tier-2b's mined attribute directions.

**Negation in vision-language models**
- Alhamoud, Alshammari, Tian, Li, Torr, Kim, Ghassemi (2025). *Vision-Language Models Do Not Understand Negation* (NegBench). CVPR. — CLIP's affirmation bias; motivates treating "−X" asymmetrically.
- Kazemi Ranjbar, Alhamoud, Ghassemi (2025). *SpaceVLM: Sub-Space Modeling of Negation in Vision-Language Models*. arXiv:2511.13231. — negation as a subspace/region ("A but not N"); direct inspiration for Tier-2a's negation penalty.
- Sammani, Chamiti, Gavrikov, Deligiannis (2026). *When Negation Is a Geometry Problem in Vision-Language Models*. arXiv:2603.20554. — a steerable negation direction exists in CLIP and can be intervened on at test time.
